# Nurse Rostering — Metaheuristic Solver (Genetic Algorithm)

This notebook develops a Genetic Algorithm (GA) for the nurse rostering problem, together with the roster evaluator used to score candidate solutions. The same evaluator scores solutions produced by both the exact (Integer Linear Programming) and the metaheuristic (GA) methods, so the two approaches are compared on an identical objective and constraint set.

Benchmark: Nurse Rostering Benchmark Instances 1–24 (Curtois & Qu). See `data/README.md`.

## 1. Roster evaluation

The evaluator is implemented independently of the exact model, so that neither method is scored by its own objective code. For any candidate roster it returns:

| Key | Meaning |
|---|---|
| `penalty` | soft objective: under-/over-cover penalties plus unmet shift-on and violated shift-off requests |
| `hard` | violation amount for each hard constraint H1–H9 (0 when satisfied) |
| `feasible` | `True` if and only if every hard violation is 0 |

**Roster representation.** A NumPy integer array of shape `(n_nurses, n_days)`, following the convention documented in `rostering_common.py`: `0` denotes a day off and `k` denotes the k-th shift type in `Instance.shift_ids` (1-based).

**Reference values (Instance 1).** The empty roster has penalty `7137`; a roster in which one nurse works the shift on days 2 and 3 has penalty `6933`. These are used to validate the evaluator.

In [13]:
import numpy as np
from rostering_common import load_instance, Instance   # shared instance parser

### 1.1 Evaluator

Each constraint is checked by a small, self-contained block so that it can be read and tested in isolation.

In [14]:
def _runs(mask):
    """Yield (start, length) of every maximal run of True in a 1-D boolean list."""
    i, n = 0, len(mask)
    while i < n:
        if mask[i]:
            j = i
            while j < n and mask[j]:
                j += 1
            yield i, j - i
            i = j
        else:
            i += 1


def evaluate(roster: np.ndarray, inst: Instance) -> dict:
    """Score one roster against one instance.

    Returns {'penalty': float, 'hard': {'H1'..'H9': amount}, 'feasible': bool}.
    The penalty is the soft objective, in the same units as the ILP objective.
    """
    n_nurses, n_days = roster.shape
    shift_of = {k + 1: sid for k, sid in enumerate(inst.shift_ids)}   # roster value -> shift id

    # ---------- Soft objective ----------
    penalty = 0.0

    # S3 cover: for each (day, shift) count the nurses working it, pay under- and over-cover
    for (d, sid), (req, w_under, w_over) in inst.cover.items():
        val = inst.shift_ids.index(sid) + 1
        worked = int(np.count_nonzero(roster[:, d] == val))
        penalty += w_under * max(0, req - worked) + w_over * max(0, worked - req)

    # S1 shift-on requests: pay the weight when the requested shift is not worked
    for nurse, d, sid, w in inst.on_requests:
        i = inst.nurse_ids.index(nurse)
        if roster[i, d] != inst.shift_ids.index(sid) + 1:
            penalty += w

    # S2 shift-off requests: pay the weight when the avoided shift is worked
    for nurse, d, sid, w in inst.off_requests:
        i = inst.nurse_ids.index(nurse)
        if roster[i, d] == inst.shift_ids.index(sid) + 1:
            penalty += w

    # ---------- Hard constraints H1..H9 (violation amount, 0 = satisfied) ----------
    hard = {f"H{k}": 0.0 for k in range(1, 10)}

    for i, nurse in enumerate(inst.nurse_ids):
        row = roster[i]
        work = [row[d] != 0 for d in range(n_days)]

        # H1 at most one shift per day: guaranteed by the encoding (one integer per cell)

        # H2 forbidden succession: the shift on day d+1 must not be forbidden after day d
        for d in range(n_days - 1):
            if row[d] != 0 and row[d + 1] != 0:
                if shift_of[row[d + 1]] in inst.forbidden_next[shift_of[row[d]]]:
                    hard["H2"] += 1

        # H3 maximum shifts per type
        for k, sid in enumerate(inst.shift_ids, start=1):
            cnt = int(np.count_nonzero(row == k))
            limit = inst.max_shifts[nurse].get(sid, n_days)   # unlisted shift -> treated as unlimited
            hard["H3"] += max(0, cnt - limit)

        # H4 total working-time window (minutes between the contractual minimum and maximum)
        minutes = sum(inst.shift_len[shift_of[row[d]]] for d in range(n_days) if row[d] != 0)
        hard["H4"] += max(0, minutes - inst.max_minutes[nurse]) + max(0, inst.min_minutes[nurse] - minutes)

        # H5 maximum consecutive working days
        for _, L in _runs(work):
            if L > inst.max_consec[nurse]:
                hard["H5"] += L - inst.max_consec[nurse]

        # H6 minimum consecutive working days (blocks touching the start or end are exempt)
        for s, L in _runs(work):
            if not (s == 0 or s + L == n_days) and L < inst.min_consec[nurse]:
                hard["H6"] += 1

        # H7 minimum consecutive days off (rest blocks; blocks touching the edges are exempt)
        off = [not w for w in work]
        for s, L in _runs(off):
            if not (s == 0 or s + L == n_days) and L < inst.min_days_off[nurse]:
                hard["H7"] += 1

        # H8 maximum weekends worked (Saturday = day 7e+5, Sunday = day 7e+6)
        wk = sum(1 for e in range(n_days // 7) if work[7 * e + 5] or work[7 * e + 6])
        hard["H8"] += max(0, wk - inst.max_weekends[nurse])

        # H9 fixed days off
        for d in inst.days_off[nurse]:
            if row[d] != 0:
                hard["H9"] += 1

    feasible = all(v == 0 for v in hard.values())
    return {"penalty": float(penalty), "hard": hard, "feasible": feasible}

### 1.2 Validation against reference values

The evaluator is checked against two independently derived penalty values on Instance 1 before it is used to drive the search.

In [15]:
inst1 = load_instance("data/raw/Instance1.txt")
print(f"Instance1: {inst1.n_nurses} nurses, {inst1.n_days} days, shift types = {inst1.shift_ids}")

# (1) empty roster: every nurse off on every day
r_empty = np.zeros((inst1.n_nurses, inst1.n_days), dtype=int)
res_empty = evaluate(r_empty, inst1)

# (2) one nurse (row 0) works the shift (value 1) on days 2 and 3
r_one = np.zeros((inst1.n_nurses, inst1.n_days), dtype=int)
r_one[0, 2] = 1
r_one[0, 3] = 1
res_one = evaluate(r_one, inst1)

print(f"empty roster                 penalty = {res_empty['penalty']:.0f}   (reference 7137)")
print(f"one nurse works days 2 and 3 penalty = {res_one['penalty']:.0f}   (reference 6933)")

assert res_empty['penalty'] == 7137, "empty-roster penalty must equal 7137"
assert res_one['penalty'] == 6933, "single-nurse penalty must equal 6933"
print("\nEvaluator validated: both reference values reproduced exactly.")

Instance1: 8 nurses, 14 days, shift types = ['D']
empty roster                 penalty = 7137   (reference 7137)
one nurse works days 2 and 3 penalty = 6933   (reference 6933)

Evaluator validated: both reference values reproduced exactly.


### 1.3 Coverage across benchmark instances

The reference values above use a single-shift instance. The following confirms that the evaluator parses and scores the full range of instance sizes used in the experiments, including the multi-shift instances.

In [16]:
for n in [1, 2, 3, 6, 8, 12]:
    inst = load_instance(f"data/raw/Instance{n}.txt")
    empty = np.zeros((inst.n_nurses, inst.n_days), dtype=int)
    res = evaluate(empty, inst)
    print(f"Instance{n:<2}  {inst.n_nurses:>3} nurses x {inst.n_days:>2} days x {inst.n_shift_types:>2} shifts   "
          f"empty-roster penalty = {res['penalty']:.0f}")

Instance1     8 nurses x 14 days x  1 shifts   empty-roster penalty = 7137
Instance2    14 nurses x 14 days x  2 shifts   empty-roster penalty = 10882
Instance3    20 nurses x 14 days x  3 shifts   empty-roster penalty = 15474
Instance6    18 nurses x 28 days x  3 shifts   empty-roster penalty = 30057
Instance8    30 nurses x 28 days x  4 shifts   empty-roster penalty = 48486
Instance12   60 nurses x 28 days x 10 shifts   empty-roster penalty = 101241
